In [1]:
import faiss
import numpy as np
import pandas as pd

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware


# ==========================================================
# Music Explorer
# ==========================================================

class MusicExplorer:

    def __init__(
        self,
        metadata: pd.DataFrame,
        embeddings: np.ndarray,
        index: faiss.Index
    ):
        self.metadata = metadata
        self.embeddings = embeddings
        self.index = index

    def find_song(
        self,
        title_name: str
    ):

        return self.metadata[
            self.metadata["track"]
            .str.contains(
                title_name,
                case=False,
                na=False
            )
        ]

    def get_song(
        self,
        row_id: int
    ):

        row = self.metadata.iloc[row_id]

        return {
            "id": int(row_id),
            "track": row["track"],
            "artist": row["artist"]
        }

    def get_neighbours(
        self,
        row_id: int,
        k: int = 10
    ):

        query = self.embeddings[
            row_id:row_id + 1
        ]

        D, I = self.index.search(
            query,
            k + 1
        )

        results = []

        for idx, score in zip(
            I[0][1:],
            D[0][1:]
        ):

            row = self.metadata.iloc[idx]

            results.append(
                {
                    "id": int(idx),
                    "track": row["track"],
                    "artist": row["artist"],
                    "score": float(score)
                }
            )

        return results

    def search(
        self,
        title_name: str,
        k: int = 10
    ):

        matches = self.find_song(
            title_name
        )

        if len(matches) == 0:

            return {
                "error": "song not found"
            }

        row_id = matches.index[0]

        return self.get_neighbours(
            row_id=row_id,
            k=k
        )

    def get_two_hop_graph(
        self,
        row_id: int,
        k: int = 5
    ):

        nodes = {}
        edges = []

        center = self.get_song(
            row_id
        )

        nodes[row_id] = {
            **center,
            "type": "query"
        }

        first_hop = self.get_neighbours(
            row_id=row_id,
            k=k
        )

        for neighbour in first_hop:

            nid = neighbour["id"]

            nodes[nid] = {
                "id": nid,
                "track": neighbour["track"],
                "artist": neighbour["artist"],
                "type": "hop_1"
            }

            edges.append(
                {
                    "source": row_id,
                    "target": nid,
                    "weight": neighbour["score"]
                }
            )

            second_hop = self.get_neighbours(
                row_id=nid,
                k=k
            )

            for second in second_hop:

                sid = second["id"]

                if sid not in nodes:

                    nodes[sid] = {
                        "id": sid,
                        "track": second["track"],
                        "artist": second["artist"],
                        "type": "hop_2"
                    }

                edges.append(
                    {
                        "source": nid,
                        "target": sid,
                        "weight": second["score"]
                    }
                )

        return {
            "nodes": list(nodes.values()),
            "edges": edges
        }


# ==========================================================
# Load Data
# ==========================================================

print("Loading metadata...")

metadata = pd.read_parquet(
    "metadata.parquet"
)

print("Loading embeddings...")

embeddings = np.load(
    "embeddings.npy"
).astype("float32")

print("Loading FAISS index...")

index = faiss.read_index(
    "music.index"
)

print("Creating explorer...")

explorer = MusicExplorer(
    metadata=metadata,
    embeddings=embeddings,
    index=index
)


# ==========================================================
# FastAPI
# ==========================================================

app = FastAPI(
    title="Music Explorer API"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ==========================================================
# Routes
# ==========================================================

@app.get("/")
def root():

    return {
        "status": "running"
    }


@app.get("/search")
def search(
    query: str,
    k: int = 10
):

    return explorer.search(
        title_name=query,
        k=k
    )


@app.get("/song")
def song(
    row_id: int
):

    return explorer.get_song(
        row_id=row_id
    )


@app.get("/graph")
def graph(
    row_id: int,
    k: int = 5
):

    return explorer.get_two_hop_graph(
        row_id=row_id,
        k=k
    )


# ==========================================================
# Run
# ==========================================================

# Save as:
#
# app.py
#
# Run:
#
# uvicorn app:app --reload
#
# Open:
#
# http://localhost:8000/docs
#
# Example:
#
# http://localhost:8000/search?query=Man%20in%20the%20Mirror
#
# http://localhost:8000/graph?row_id=123

ModuleNotFoundError: No module named 'fastapi'